In [ ]:
import pandas as pd

# 1. DATA LOADING

voters = pd.read_excel('VotersPublic.xlsx')
patients = pd.read_excel('PatientsPublic.xlsx')
patients_kanon = pd.read_excel('PatientsPublicKAnon.xlsx')

# 2. LINKAGE ATTACK IMPLEMENTATION

# Performing a relational join between datasets using Quasi-Identifiers (ZIP code, Age, Sex)
linkage_attack = pd.merge(voters, patients, on=['ZIP code', 'Age', 'Sex'])

print("=" * 50)
print("📋 PHASE 1: LINKAGE ATTACK RESULTS")
print("=" * 50 + "\n")

if not linkage_attack.empty:
    # Displaying identified citizens' personal data along with their sensitive medical diagnosis
    print(linkage_attack[['First Name_x', 'Last Name_x', 'Age', 'Sex', 'ZIP code', 'Diagnosis']])
    print("\n" + "-" * 40)
else:
    print("No automatic identifications found.")

print(f"🔹 Total potential matches generated: {len(linkage_attack)}")
unique_identified_voters = linkage_attack[['First Name_x', 'Last Name_x']].drop_duplicates()
print(f"🚨 Number of unique individuals successfully re-identified: {len(unique_identified_voters)}\n")



# 3. DEFENSE EVALUATION (k-Anonymized Data)

print("=" * 50)
print("🛡️ PHASE 2: EVALUATION OF k-ANONYMIZED DATA")
print("=" * 50 + "\n")

# Converting columns to strings to prevent parsing errors caused by generalized age brackets (e.g., '[50-67]')
voters['ZIP code'] = voters['ZIP code'].astype(str)
voters['Age'] = voters['Age'].astype(str)
patients_kanon['Age'] = patients_kanon['Age'].astype(str)
patients_kanon['ZIP code'] = patients_kanon['ZIP code'].astype(str)

# Executing the exact same linkage attack on the protected dataset
kanon_check = pd.merge(voters, patients_kanon, on=['ZIP code', 'Age', 'Sex'])

print(f"🔹 Total matches found after k-anonymization: {len(kanon_check)}")

# Mathematical validation of data anonymity
if len(kanon_check) > 0:
    # Calculating the density of each Equivalence Class
    # If k-Anonymity is successful, every individual must be indistinguishable from a group of at least k individuals
    group_sizes = kanon_check.groupby(['ZIP code', 'Age', 'Sex']).size()
    min_k = group_sizes.min()

    print(f"✔️ Identification is NOT possible.")
    print(f"🔒 Security Proof: Every victim is now hidden within an equivalence class of at least k = {min_k} individuals.")
else:
    print("✔️ Safe: Zero matches found.")
print("\n" + "=" * 50)